# NAMD-FEP — notebook 5 · equilibration checks and the free energy

The last step: from per-window `.fepout` files to **ΔΔG**, with the checks that
make the number believable. Everything here is read-only and works on the shipped
6I5I example, so this notebook runs end to end on Binder even without a GPU.

The questions, in order:

0. **minimisation & equilibration** — did the system settle before production?
1. **parser validation** — does our extraction reproduce NAMD's own per-window ΔG?
2. **BAR per window** — the estimate itself, complex minus solvent.
3. **hysteresis** — do forward and backward agree? (reversibility)
4. **stationarity** — is each window converged within itself? (equilibration drift)
5. **bootstrap error bars** and the **trim-sensitivity scan**.

The shipped example's own verdict (from the README): *reversible but not converged*
— the two directions agree, but the complex leg drifts. This notebook re-derives
that conclusion from the raw output, figure by figure.

## 5.0 Setup — and did the system equilibrate?

Each leg's production is preceded by minimisation and NVT/NPT equilibration
(`nvt_equil.namd`, `npt_equil.namd`). NAMD writes its own `FepEnergy` rows for those
stages too (the alchemical flags are on), so the **dE trace through equilibration**
is a direct, on-disk record of the system settling: it should level off, not trend.

The second check is the per-window trace: every production window opens with
`alchEquilSteps` of equilibration (the first 99 of 500 rows here) which the analysis
**discards** — the plot shows why that trim exists.

In [ ]:
import os, subprocess, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
if not (REPO / "bin" / "rbfe").exists():
    REPO = REPO.parent
os.environ["PATH"] = str(REPO / "bin") + os.pathsep + os.environ["PATH"]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from rbfe import analysis as A
from rbfe.config import load as load_config

# The tutorial system if it has run output, else the shipped 6I5I example.
TUTORIAL = REPO / "_tutorial" / "6i5i"
LEGACY   = REPO / "6I5I_DUAL_FEP"
SHIM     = REPO / "_tutorial" / "legacy_6i5i"

def has_output(d):
    return (d / "complex").exists() and any((d / "complex").glob("*_w00.fepout"))

if (TUTORIAL / "system.ini").exists() and has_output(TUTORIAL):
    SYS, ROOT = TUTORIAL, TUTORIAL
elif (SHIM / "system.ini").exists():
    SYS, ROOT = SHIM, LEGACY            # config shim written by notebook 04
else:
    raise SystemExit("run notebook 04 first (it writes the legacy 6I5I shim)")

cfg = load_config(SYS / "system.ini")
A.configure(cfg)
print(f"{cfg.name}: {cfg.nwin} windows, T = {cfg.get('run','temperature')} K")
print("run output:", ROOT)

# ---- the equilibration traces -----------------------------------------
fig, axes = plt.subplots(1, len(cfg.legs), figsize=(6 * len(cfg.legs), 3.6),
                         squeeze=False)
found = False
for ax, leg in zip(axes[0], cfg.legs):
    for stage, colour in (("nvt_equil", "tab:purple"), ("npt_equil", "tab:orange")):
        f = ROOT / leg / f"{stage}.fepout"
        if not f.exists():
            continue
        dE = [float(l.split()[6]) for l in f.read_text().splitlines()
              if l.startswith("FepEnergy")]
        if not dE:
            continue
        found = True
        n = len(dE)
        half = n // 2
        ax.plot(dE, lw=0.4, color=colour, alpha=0.6)
        # a coarse running mean makes a trend visible through the noise
        k = max(n // 50, 5)
        ax.plot(np.convolve(dE, np.ones(k) / k, "valid"), color=colour,
                label=f"{stage}: half means {np.mean(dE[:half]):+.3f} -> "
                      f"{np.mean(dE[half:]):+.3f}")
    ax.set_title(f"{leg}: dE during equilibration")
    ax.set_xlabel("output row")
    ax.legend(frameon=False, fontsize=8)
axes[0][0].set_ylabel("dE (kcal/mol)")
plt.tight_layout()
if found:
    plt.show()
    print("read: a settled system has a flat running mean and similar half-means.")
else:
    print("no equilibration .fepout files on disk for this system")

The per-window picture — why the first 99 of every 500 rows are thrown away:

In [ ]:
# One production window, shown in full: equilibration rows (red, discarded) vs
# production rows (green, used). NAMD's own accumulator makes the same cut.
f = None
for leg in cfg.legs:
    cand = ROOT / leg / "md_forward_w05.fepout"
    if cand.exists():
        f = cand
        break

if f is None:
    print("no per-window production files yet")
else:
    w = A.parse_windows(f)[0]
    allrows, start = w["all"], w["prod_start"]
    plt.figure(figsize=(8, 3.4))
    plt.plot(range(start), allrows[:start], ".", color="tab:red", ms=3,
             label=f"equilibration (rows 0-{start-1}, discarded)")
    plt.plot(range(start, len(allrows)), allrows[start:], ".",
             color="tab:green", ms=3,
             label=f"production (rows {start}+, used)")
    plt.axvline(start - 0.5, color="k", lw=0.8, ls="--")
    plt.title(f"{f.parent.name}/{f.name}  lambda {w['l1']} -> {w['l2']}")
    plt.xlabel("row in window")
    plt.ylabel("dE (kcal/mol)")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()

## 5.1 Load all four directions through the audit's parser

`collect_leg` reads the **per-window** files (never the assembled ones — assembly
strips the comment lines the validation needs), salvaging from the single-run file
where a window has none. `trim=None` applies the auto-detected equilibration cut.

In [ ]:
legs = {}
for leg in A.LEGS:
    legs[leg] = {}
    for d, direction in (("fwd", "forward"), ("bwd", "backward")):
        legs[leg][d] = A.collect_leg(ROOT, leg, direction)
        n = sum(len(w["dE"]) for w in legs[leg][d])
        print(f"{leg:8s} {direction:9s} {len(legs[leg][d])} windows, "
              f"{n} production samples")

## 5.2 Parser validation — our EXP vs NAMD's own per-window ΔG

NAMD writes its own per-window free energy into the `.fepout`. Recomputing the same
number from the raw dE column and matching it to ~10⁻⁶ kcal/mol proves the dE
column, the window splitting, and the equilibration trim are all correct. **This is
the audit's section A.**

In [ ]:
A.section_A(legs)

## 5.3 The estimate — BAR per window, both legs

Bennett's acceptance ratio between the forward and backward samples of each window,
summed along the λ schedule, per leg. **ΔΔG = ΔG_complex − ΔG_solvent.**

In [ ]:
tc, ts, ddg = A.section_B(legs)

# the same numbers, as the README banner draws them
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
for leg, colour in (("complex", "tab:blue"), ("solvent", "tab:red")):
    dg, per = A.bar_leg(legs[leg]["fwd"], legs[leg]["bwd"])
    xs = [w["l1"] for w in legs[leg]["fwd"]]
    ax1.plot(xs, per, "o-", color=colour, ms=4, label=f"{leg} ({dg:+.3f})")
ax1.axhline(0, color="grey", lw=0.8)
ax1.set_xlabel("lambda (window start)")
ax1.set_ylabel("per-window ΔG (kcal/mol)")
ax1.legend(frameon=False)
ax1.set_title("BAR per window")

per_c = A.bar_leg(legs["complex"]["fwd"], legs["complex"]["bwd"])[1]
per_s = A.bar_leg(legs["solvent"]["fwd"], legs["solvent"]["bwd"])[1]
xs = [w["l1"] for w in legs["complex"]["fwd"]]
ax2.plot(xs, np.cumsum([c - s for c, s in zip(per_c, per_s)]),
         "o-", color="k", ms=4)
ax2.axhline(0, color="grey", lw=0.8)
ax2.set_xlabel("lambda (window start)")
ax2.set_ylabel("cumulative ΔΔG (kcal/mol)")
ax2.set_title(f"converging to ΔΔG = {ddg:+.3f} kcal/mol")
plt.tight_layout()
plt.show()

## 5.4 Hysteresis — do the two directions agree?

Forward and backward are independent estimates of the same window, so EXP(fwd) +
EXP(bwd) should be ≈ 0 per window and overall. This is the **reversibility** check.

In [ ]:
A.section_C(legs)

fig, ax = plt.subplots(figsize=(8, 3.4))
for leg, colour in (("complex", "tab:blue"), ("solvent", "tab:red")):
    ef, per_f = A.exp_leg(legs[leg]["fwd"])
    eb, per_b = A.exp_leg(legs[leg]["bwd"])
    # backward windows run high->low lambda; pair window i with bwd[N-1-i]
    hyst = [f + b for f, b in zip(per_f, per_b[::-1])]
    xs = [w["l1"] for w in legs[leg]["fwd"]]
    ax.bar([x - 0.008 for x in xs] if leg == "complex" else [x + 0.008 for x in xs],
           hyst, width=0.016, color=colour, alpha=0.75,
           label=f"{leg} (sum {ef + eb:+.3f})")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("lambda (window start)")
ax.set_ylabel("hysteresis per window (kcal/mol)")
ax.legend(frameon=False)
ax.set_title("0 = perfectly reversible")
plt.tight_layout()
plt.show()

## 5.5 Stationarity — has each window equilibrated?

Split each window in half and BAR the halves against each other: a converged,
equilibrated leg sums to ~0. The shipped 6I5I run is the cautionary case — its
complex leg carries systematic drift, which is why the README calls the result
*reversible but not converged*.

In [ ]:
A.section_E(legs)

fig, ax = plt.subplots(figsize=(8, 3.4))
for j, (leg, colour) in enumerate((("complex", "tab:blue"), ("solvent", "tab:red"))):
    drifts, xs = [], []
    for i in range(A.NWIN):
        wf = legs[leg]["fwd"][i]
        wb = legs[leg]["bwd"][A.NWIN - 1 - i]
        h = len(wf["dE"]) // 2
        drifts.append(A.bar_np(wf["dE"][h:], -wb["dE"][h:])
                      - A.bar_np(wf["dE"][:h], -wb["dE"][:h]))
        xs.append(wf["l1"])
    ax.bar([x - 0.008 + 0.016 * j for x in xs], drifts, width=0.016,
           color=colour, alpha=0.75, label=f"{leg} (sum {sum(drifts):+.3f})")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("lambda (window start)")
ax.set_ylabel("late − early half ΔG (kcal/mol)")
ax.legend(frameon=False)
ax.set_title("stationarity: 0 = converged within each window")
plt.tight_layout()
plt.show()

## 5.6 Error bars — moving-block bootstrap

Resamples the dE time series in blocks (default 25 samples = 25 ps) and re-computes
the whole BAR chain 400 times. Costs a couple of minutes; set `SKIP_BOOT` for the
fast version.

In [ ]:
SKIP_BOOT = False   # True for the quick pass

if SKIP_BOOT:
    print("skipped (SKIP_BOOT=True)")
else:
    A.section_D(legs, blocks=25, nboot=400)

## 5.7 Sensitivity — ΔΔG vs the equilibration trim

How much does the answer move if more (or less) of each window's head is discarded?
A stable ΔΔG across trims is the practical definition of *equilibrated enough*; a
monotone drift is what unconverged looks like. Trim values are multiples of the
auto-detected cut.

In [ ]:
e = cfg.equil_rows
trims = sorted({0, e // 2, e, int(e * 1.5), int(e * 2), int(e * 3)})
A.run_scan(ROOT, trims)

# same table, drawn
ddgs, ns = [], []
for trim in trims:
    lg = {leg: {d: A.collect_leg(ROOT, leg, "forward" if d == "fwd" else "backward",
                                 trim)
                for d in ("fwd", "bwd")}
          for leg in ("complex", "solvent")}
    ddgs.append(A.bar_pair(lg, "complex")[0] - A.bar_pair(lg, "solvent")[0])
    ns.append(len(lg["complex"]["fwd"][0]["dE"]))

plt.figure(figsize=(6.5, 3.4))
plt.plot(trims, ddgs, "o-", color="k")
plt.axvline(e, color="tab:red", ls="--", lw=0.8,
            label=f"auto-detected trim ({e} rows)")
plt.xlabel("rows discarded per window")
plt.ylabel("ΔΔG (kcal/mol)")
plt.title("trim sensitivity")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

## 5.8 Reading the verdict

The four numbers to hold together, exactly as the README states them for 6I5I:

* **ΔΔG and its CI** (5.3, 5.6) — is the interval tight enough to rank anything?
* **hysteresis** (5.4) — inside the statistical error → the simulation is
  *reversible*;
* **stationarity** (5.5) — drift ≫ error → a leg has **not equilibrated**, and the
  point estimate is not a measured binding difference;
* **trim sensitivity** (5.7) — the same story told a second way.

For the shipped 6I5I run that reads: ΔΔG ≈ −0.106 kcal/mol, CI straddling zero,
hysteresis within error, but +0.365 kcal/mol of drift on the complex leg —
**reversible but not converged: a null result, not a measured ΔΔG.** A production
re-run with longer windows (the `systems/6i5i/system.ini` schedule doubles them to
500 000 steps) is the fix, and notebooks 03–04 are how you would launch it.

The command-line equivalent of this whole notebook, on any system:

```bash
cd <system> && rbfe audit            # + --scan for 5.7, --skip-boot for the fast pass
```